In [1]:
import folium
import random
import matplotlib.colors as mcolors
from folium.features import CustomIcon

def extract_plot_data_by_MMSI(AIS_data, MMSI_list, RF_data=None, include_AIS=True, include_RF=False):
    """
    Extracts AIS and/or RF data for a given list of MMSIs.
    
    Args:
        AIS_data (pd.DataFrame): AIS data
        RF_data (pd.DataFrame): RF data, defaults to None
        MMSI_list (list): List of MMSI numbers to filter by
        include_AIS (bool): Whether to include AIS data, defaults to True
        include_RF (bool): Whether to include RF data, defaults to False

    Returns:
        tuple: Filtered AIS and RF data
    """
    filtered_AIS = None
    filtered_RF = None

    if include_AIS and AIS_data is not None:
        filtered_AIS = AIS_data[AIS_data["MMSI"].isin(MMSI_list)]

    if include_RF and RF_data is not None:
        filtered_RF = RF_data[RF_data["MMSI"].isin(MMSI_list)]

    return filtered_AIS, filtered_RF

def map_vessel_route(AIS_data, MMSI_list, RF_data=None, include_AIS=True, include_RF=False): 
    """
    Plots the routes of vessels on an interactive map using Folium.

    Args:
        AIS_data (pd.DataFrame): AIS data
        RF_data (pd.DataFrame): RF data, defaults to None
        MMSI_list (list): List of MMSI numbers to filter by
        include_AIS (bool): Whether to include AIS data points in the plot, defaults to True
        include_RF (bool): Whether to include RF data points in the plot, defaults to False

    Returns:
        folium.Map: Interactive folium map
    """
    # Get the filtered data
    AIS_data, RF_data = extract_plot_data_by_MMSI(AIS_data, MMSI_list, RF_data, include_AIS, include_RF)
    
    # Determine center of the map
    if AIS_data is not None and not AIS_data.empty:
        map_center = [AIS_data["LAT"].mean(), AIS_data["LON"].mean()]
    elif RF_data is not None and not RF_data.empty:
        map_center = [RF_data["RF_LAT"].mean(), RF_data["RF_LON"].mean()]
    else:
        return folium.Map(location=[0, 0], zoom_start=2)

    # Create map
    map = folium.Map(location=map_center, zoom_start=7, tiles="CartoDB Positron")
    colors = list(mcolors.CSS4_COLORS.keys()) 

    # Plot AIS data
    if AIS_data is not None and not AIS_data.empty:
        for i, (mmsi, group) in enumerate(AIS_data.groupby('MMSI')):
            AIS_filter = folium.FeatureGroup(name=f"AIS data points {mmsi}")
            color = random.choice(colors)
            
            for _, row in group.iterrows():
                heading = 315 + row.get('Heading', 0)
                icon_html = f'''
                    <div style="font-size:12px; transform: rotate({heading}deg);">
                        <i class="fa-solid fa-location-arrow" style="color: {color};"></i>
                    </div>
                    '''
                folium.Marker(
                    location=(row['LAT'], row['LON']),
                    tooltip=f"AIS data point<br>MMSI: {mmsi}<br>Timestamp: {row['BaseDateTime']}",
                    icon=folium.DivIcon(html=icon_html)
                ).add_to(AIS_filter)
                
            AIS_filter.add_to(map)

    # Plot RF data
    if RF_data is not None and not RF_data.empty:
        RF_filter = folium.FeatureGroup(name="RF data points")
        
        for (mmsi, timestamp), group in RF_data.groupby(['MMSI', 'TimeStamp_RF']):
            for _, row in group.iterrows():
                custom_icon = CustomIcon(
                    icon_image="../images/star_image_yellow.png",
                    icon_size=(30, 30),
                    icon_anchor=(15, 15)
                )
                folium.Marker(
                    location=[row['RF_LAT'], row['RF_LON']],
                    icon=custom_icon,
                    tooltip=f"RF data point<br>MMSI: {mmsi}<br>Timestamp: {row['TimeStamp_RF']}",
                ).add_to(RF_filter)
                
        RF_filter.add_to(map)

    # Add layer control
    folium.LayerControl(collapsed=False).add_to(map)

    return map


In [2]:
import folium
import random
import matplotlib.colors as mcolors
from folium.features import CustomIcon

def map_vessel_route_2(AIS_data, MMSI_list, RF_data=None, include_AIS=True, include_RF=False): 
    """
    Plots the routes of vessels on an interactive map using Folium.

    Args:
        AIS_data (pd.DataFrame): AIS data
        RF_data (pd.DataFrame): RF data, defaults to None
        MMSI_list (list): List of MMSI numbers to filter by
        include_AIS (bool): Whether to include AIS data points in the plot, defaults to True
        include_RF (bool): Whether to include RF data points in the plot, defaults to False

    Returns:
        folium.Map: Interactive folium map
    """
    # Get filtered data
    AIS_data, RF_data = extract_plot_data_by_MMSI(AIS_data, MMSI_list, RF_data, include_AIS, include_RF)
    
    # Determine map center
    if AIS_data is not None and not AIS_data.empty:
        map_center = [AIS_data["LAT"].mean(), AIS_data["LON"].mean()]
    elif RF_data is not None and not RF_data.empty:
        map_center = [RF_data["RF_LAT"].mean(), RF_data["RF_LON"].mean()]
    else:
        return folium.Map(location=[0, 0], zoom_start=2)

    # Create map
    map = folium.Map(location=map_center, zoom_start=7, tiles="CartoDB Positron")
    colors = list(mcolors.CSS4_COLORS.keys())

    # Plot AIS data with color per SubTrackID
    if AIS_data is not None and not AIS_data.empty:
        for (mmsi, subtrack), group in AIS_data.groupby(['MMSI', 'route_id']):
            AIS_filter = folium.FeatureGroup(name=f"AIS MMSI: {mmsi}, Route ID: {subtrack}")
            color = random.choice(colors)
            
            for _, row in group.iterrows():
                heading = 315 + row.get('Heading', 0)
                icon_html = f'''
                    <div style="font-size:12px; transform: rotate({heading}deg);">
                        <i class="fa-solid fa-location-arrow" style="color: {color};"></i>
                    </div>
                    '''
                folium.Marker(
                    location=(row['LAT'], row['LON']),
                    tooltip=f"AIS data point<br>MMSI: {mmsi}<br>Route ID: {subtrack}<br>Timestamp: {row['BaseDateTime']}",
                    icon=folium.DivIcon(html=icon_html)
                ).add_to(AIS_filter)
            
            AIS_filter.add_to(map)

    # Plot RF data (optioneel)
    if RF_data is not None and not RF_data.empty:
        RF_filter = folium.FeatureGroup(name="RF data points")
        
        for (mmsi, timestamp), group in RF_data.groupby(['MMSI', 'TimeStamp_RF']):
            for _, row in group.iterrows():
                custom_icon = CustomIcon(
                    icon_image="../images/star_image_yellow.png",
                    icon_size=(30, 30),
                    icon_anchor=(15, 15)
                )
                folium.Marker(
                    location=[row['RF_LAT'], row['RF_LON']],
                    icon=custom_icon,
                    tooltip=f"RF data point<br>MMSI: {mmsi}<br>Timestamp: {row['TimeStamp_RF']}",
                ).add_to(RF_filter)
        
        RF_filter.add_to(map)

    # Layer control
    folium.LayerControl(collapsed=False).add_to(map)

    return map


In [3]:
map_vessel_route(time_df, [563935000])

NameError: name 'time_df' is not defined

In [ ]:
map_vessel_route_2(df_test_filtered, [563935000])